# End-to-End Knowledge Graph RAG: Roman Empire

This notebook demonstrates building an end-to-end RAG system using knowledge graphs with Neo4j and LangChain.

The system:
1. Loads Wikipedia data about the Roman Empire
2. Extracts entities and relationships using LLMs
3. Stores them in a Neo4j graph database
4. Implements hybrid retrieval (vector + graph) for RAG

## 1. Import Dependencies

In [ ]:
from dotenv import load_dotenv
import os
from typing import Tuple, List

from langchain_community.graphs import Neo4jGraph
from langchain_community.document_loaders import WikipediaLoader
from langchain_community.vectorstores import Neo4jVector
from langchain_community.vectorstores.neo4j_vector import remove_lucene_chars

from langchain_text_splitters import TokenTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_experimental.graph_transformers import LLMGraphTransformer
from databricks_langchain import DatabricksEmbeddings

from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate
from pydantic import BaseModel, Field
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser

## 2. Load Environment Variables

In [ ]:
load_dotenv()

AURA_INSTANCENAME = os.environ["AURA_INSTANCENAME"]
NEO4J_URI = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]
AUTH = (NEO4J_USERNAME, NEO4J_PASSWORD)

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# OPENAI_ENDPOINT = os.getenv("OPENAI_ENDPOINT")

## 3. Initialize LLM and Neo4j Graph Connection

In [ ]:
# ============================================================================
# SETUP: Import LLM Helper Functions
# ============================================================================
# We use helper functions to create LLM instances with proper configuration
# These functions handle API key loading and model configuration

import os
import sys

# Add parent directory to path for importing helpers
sys.path.append(os.path.abspath("../.."))

# Import our LLM factory functions
# - get_groq_llm(): Creates a Groq-hosted LLM (fast inference)
# - get_openai_llm(): Creates an OpenAI GPT model
from helpers.utils import get_groq_llm, get_openai_llm,get_databricks_llm

print("LLM helpers imported successfully!")

# ============================================================================
# CREATE THE LLM AND CHATBOT GRAPH
# ============================================================================

# -----------------------------------------------------------------------------
# Step 1: Initialize the LLM
# We use Groq for fast inference, but you can swap to OpenAI
# -----------------------------------------------------------------------------
chat = get_databricks_llm("databricks-gemini-2-5-pro")  # Fast, open-source models hosted by Groq
# Alternative: llm = get_openai_llm()  # OpenAI's GPT models

if hasattr(chat, 'model_name'):
    print(f"LLM initialized: {chat.model_name}")
elif hasattr(chat, 'model'):
    print(f"LLM initialized: {chat.model} (Databricks)")
else:
    print("LLM initialized: Groq LLM")

In [ ]:
# chat = ChatOpenAI(api_key=OPENAI_API_KEY, temperature=0, model="gpt-3.5-turbo")

kg = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
)

## 4. Load and Process Wikipedia Documents

In [ ]:
# Read the Wikipedia page for the Roman Empire
raw_documents = WikipediaLoader(query="The Roman empire").load()
print(f"Loaded {len(raw_documents)} documents")

## 5. Define Chunking Strategy and Split Documents

In [ ]:
text_splitter = TokenTextSplitter(chunk_size=512, chunk_overlap=24)
documents = text_splitter.split_documents(raw_documents[:3])
print(f"Created {len(documents)} document chunks")
len(documents)

In [ ]:
documents

## 6. Convert Documents to Graph Using LLM

In [ ]:
llm_transformer = LLMGraphTransformer(llm=chat)
graph_documents = llm_transformer.convert_to_graph_documents(documents)
print(f"Created {len(graph_documents)} graph documents")

In [ ]:
graph_documents

## 7. Store Graph Documents in Neo4j

In [ ]:
res = kg.add_graph_documents(
    graph_documents,
    include_source=True,
    baseEntityLabel=True,
)
print("Graph documents stored in Neo4j")

## 8. Create Hybrid Vector Index

In [ ]:
# Use DatabricksEmbeddings for embedding model (not ChatDatabricks which is for chat)
embedding_model = DatabricksEmbeddings(endpoint="databricks-gte-large-en")

In [ ]:
vector_index = Neo4jVector.from_existing_graph(
    embedding_model,
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)
print("Vector index created")

In [ ]:
res = kg.query(
    """
  SHOW VECTOR INDEXES
  """
)
res

## 9. Define Entity Extraction Model

In [ ]:
class Entities(BaseModel):
    """Identifying information about entities."""

    names: List[str] = Field(
        ...,
        description="All the person, organization, or business entities that "
        "appear in the text",
    )

## 10. Create Entity Extraction Chain

In [ ]:
entity_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are extracting organization and person entities from the text.",
        ),
        (
            "human",
            "Use the given format to extract information from the following "
            "input: {question}",
        ),
    ]
)
entity_chain = entity_prompt | chat.with_structured_output(Entities)

## 11. Test Entity Extraction (Optional)

In [ ]:
# Test entity extraction
res = entity_chain.invoke(
    {"question": "What was The Byzantine Empire."}
).names
print(res)

## 12. Create Fulltext Index for Entity Search

In [ ]:
kg.query("CREATE FULLTEXT INDEX entity IF NOT EXISTS FOR (e:__Entity__) ON EACH [e.id]")
print("Fulltext index created")

## 13. Generate Fulltext Query Function

In [ ]:
def generate_full_text_query(input: str) -> str:
    """
    Generate a full-text search query for a given input string.

    This function constructs a query string suitable for a full-text search.
    It processes the input string by splitting it into words and appending a
    similarity threshold (~2 changed characters) to each word, then combines
    them using the AND operator. Useful for mapping entities from user questions
    to database values, and allows for some misspellings.
    """
    full_text_query = ""
    words = [el for el in remove_lucene_chars(input).split() if el]
    for word in words[:-1]:
        full_text_query += f" {word}~2 AND"
    full_text_query += f" {words[-1]}~2"
    return full_text_query.strip()

## 14. Structured Retriever Function

In [ ]:
def structured_retriever(question: str) -> str:
    """
    Collects the neighborhood of entities mentioned in the question.
    Uses fulltext search to find entities and retrieves their relationships.
    """
    result = ""
    entities = entity_chain.invoke({"question": question})
    for entity in entities.names:
        print(f" Getting Entity: {entity}")
        response = kg.query(
            """CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})
            YIELD node,score
            CALL {
              WITH node
              MATCH (node)-[r:!MENTIONS]->(neighbor)
              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
              UNION ALL
              WITH node
              MATCH (node)<-[r:!MENTIONS]-(neighbor)
              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS output
            }
            RETURN output LIMIT 50
            """,
            {"query": generate_full_text_query(entity)},
        )
        result += "\n".join([el["output"] for el in response])
    return result

## 15. Test Structured Retriever (Optional)

In [ ]:
# Test structured retriever
# print(structured_retriever("Who is Octavian?"))

## 16. Final Retriever Function (Hybrid: Structured + Unstructured)

In [ ]:
def retriever(question: str):
    """
    Hybrid retriever that combines structured (graph) and unstructured (vector) data.
    """
    print(f"Search query: {question}")
    structured_data = structured_retriever(question)
    unstructured_data = [
        el.page_content for el in vector_index.similarity_search(question)
    ]
    final_data = f"""Structured data:
{structured_data}
Unstructured data:
{"#Document ". join(unstructured_data)}
    """
    print(f"\nFinal Data::: ==>{final_data}")
    return final_data

## 17. Define Chat History Formatting Function

In [ ]:
def _format_chat_history(chat_history: List[Tuple[str, str]]) -> List:
    """Format chat history into a list of HumanMessage and AIMessage objects."""
    buffer = []
    for human, ai in chat_history:
        buffer.append(HumanMessage(content=human))
        buffer.append(AIMessage(content=ai))
    return buffer

## 18. Define Condense Question Prompt

In [ ]:
# Condense a chat history and follow-up question into a standalone question
_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question,
in its original language.
Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""

CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template(_template)

## 19. Define Search Query Runnable with Chat History Handling

In [ ]:
_search_query = RunnableBranch(
    # If input includes chat_history, we condense it with the follow-up question
    (
        RunnableLambda(lambda x: bool(x.get("chat_history"))).with_config(
            run_name="HasChatHistoryCheck"
        ),  # Condense follow-up question and chat into a standalone_question
        RunnablePassthrough.assign(
            chat_history=lambda x: _format_chat_history(x["chat_history"])
        )
        | CONDENSE_QUESTION_PROMPT
        | ChatOpenAI(temperature=0)
        | StrOutputParser(),
    ),
    # Else, we have no chat history, so just pass through the question
    RunnableLambda(lambda x: x["question"]),
)

## 20. Define Answer Generation Prompt

In [ ]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
Use natural language and be concise.
Answer:"""

answer_prompt = ChatPromptTemplate.from_template(template)

## 21. Build the Complete RAG Chain

In [ ]:
chain = (
    RunnableParallel(
        {
            "context": _search_query | retriever,
            "question": RunnablePassthrough(),
        }
    )
    | answer_prompt
    | chat
    | StrOutputParser()
)

## 22. Test the RAG Chain - Simple Question

In [ ]:
res_simple = chain.invoke(
    {
        "question": "How did the Roman empire fall?",
    }
)

print(f"\n Results === {res_simple}\n\n")

## 23. Test the RAG Chain - With Chat History (Optional)

In [ ]:
# Test with chat history for context-aware follow-up questions
# res_hist = chain.invoke(
#     {
#         "question": "When did he become the first emperor?",
#         "chat_history": [
#             ("Who was the first emperor?", "Augustus was the first emperor.")
#         ],
#     }
# )
# print(f"\n === {res_hist}\n\n")